# Step 5A — Initialize the unsupervised-learning notebook

# State-Level Economic Similarity and Structural Stability

## Unsupervised Machine-Learning Analysis

This notebook uses principal component analysis and clustering to examine
which U.S. states have similar economic and industry-employment structures.

The analysis compares three periods:

- **Baseline:** 2015–2019
- **Economic shock:** 2020–2022
- **Post-shock:** 2023–2024

The primary clustering model uses eight structural features. Four annual
growth indicators are used separately to interpret how the state economies
changed during and after the economic shock.

In [ ]:
# ============================================================
# STEP 5A-1 — IMPORT LIBRARIES
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import (
    dendrogram,
    linkage
)

from sklearn import __version__ as sklearn_version

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn_version)

### Library setup

The notebook imports libraries for data management, visualization,
standardization, dimensionality reduction, clustering, and cluster
evaluation.

A fixed random state of 42 is used to make the K-Means results reproducible.

In [ ]:
# ============================================================
# STEP 5A-2 — DEFINE DATA PATHS
# ============================================================

PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

data_paths = {
    "structural_panel":
        PROCESSED_DIR /
        "state_structural_panel_core_2015_2024.csv",

    "structural_period":
        PROCESSED_DIR /
        "state_structural_period_averages.csv",

    "growth_panel":
        PROCESSED_DIR /
        "state_dynamic_growth_panel_2015_2024.csv",

    "growth_period":
        PROCESSED_DIR /
        "state_dynamic_growth_period_averages.csv"
}

print("Project directory:", PROJECT_DIR)
print("Processed-data directory:", PROCESSED_DIR)

for dataset_name, file_path in data_paths.items():
    print(
        f"{dataset_name}:",
        "Found" if file_path.exists() else "Missing"
    )

In [ ]:
# ============================================================
# STEP 5A-3 — LOAD PROCESSED DATA
# ============================================================

missing_files = [
    str(file_path)
    for file_path in data_paths.values()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following processed files are missing:\n"
        + "\n".join(missing_files)
    )

structural_panel = pd.read_csv(
    data_paths["structural_panel"],
    dtype={"state_fips": "string"}
)

structural_period = pd.read_csv(
    data_paths["structural_period"],
    dtype={"state_fips": "string"}
)

growth_panel = pd.read_csv(
    data_paths["growth_panel"],
    dtype={"state_fips": "string"}
)

growth_period = pd.read_csv(
    data_paths["growth_period"],
    dtype={"state_fips": "string"}
)

# Preserve two-digit state FIPS codes
for dataframe in [
    structural_panel,
    structural_period,
    growth_panel,
    growth_period
]:
    dataframe["state_fips"] = (
        dataframe["state_fips"]
        .str.zfill(2)
    )

print("All processed datasets loaded.")

In [ ]:
# ============================================================
# STEP 5A-4 — DEFINE FEATURE GROUPS
# ============================================================

STRUCTURAL_FEATURES = [
    "real_gdp_per_capita",
    "real_personal_income_per_capita_2017",
    "real_output_per_job",
    "real_average_wages_salaries_2017",
    "manufacturing_share",
    "professional_business_share",
    "healthcare_share",
    "natural_resources_agriculture_share"
]

GROWTH_FEATURES = [
    "real_gdp_per_capita_growth",
    "real_income_per_capita_growth",
    "total_employment_growth",
    "real_average_wage_growth"
]

PERIOD_ORDER = [
    "Baseline_2015_2019",
    "Shock_2020_2022",
    "Post_Shock_2023_2024"
]

print("Structural features:", len(STRUCTURAL_FEATURES))
print("Growth features:", len(GROWTH_FEATURES))

In [ ]:
# ============================================================
# STEP 5A-5 — LOADING SUMMARY
# ============================================================

datasets = {
    "Structural panel": structural_panel,
    "Structural period averages": structural_period,
    "Growth panel": growth_panel,
    "Growth period averages": growth_period
}

loading_summary = pd.DataFrame([
    {
        "Dataset": dataset_name,
        "Rows": dataframe.shape[0],
        "Columns": dataframe.shape[1],
        "States": dataframe["state_fips"].nunique(),
        "First_Year": (
            dataframe["year"].min()
            if "year" in dataframe.columns
            else "Period averages"
        ),
        "Last_Year": (
            dataframe["year"].max()
            if "year" in dataframe.columns
            else "Period averages"
        ),
        "Missing_Cells": int(
            dataframe.isna().sum().sum()
        )
    }
    for dataset_name, dataframe in datasets.items()
])

display(loading_summary)

In [ ]:
# ============================================================
# STEP 5A-6 — VALIDATE LOADED DATA
# ============================================================

# Required feature columns
missing_structural_features = [
    feature
    for feature in STRUCTURAL_FEATURES
    if feature not in structural_period.columns
]

missing_growth_features = [
    feature
    for feature in GROWTH_FEATURES
    if feature not in growth_period.columns
]

if missing_structural_features:
    raise KeyError(
        "Missing structural features: "
        f"{missing_structural_features}"
    )

if missing_growth_features:
    raise KeyError(
        "Missing growth features: "
        f"{missing_growth_features}"
    )

# Panel structure
assert structural_panel.shape[0] == 500
assert structural_panel["state_fips"].nunique() == 50
assert structural_panel["year"].min() == 2015
assert structural_panel["year"].max() == 2024

assert not structural_panel.duplicated(
    ["state_fips", "year"]
).any()

# Period structure
assert structural_period.shape[0] == 150
assert structural_period["state_fips"].nunique() == 50
assert structural_period["period"].nunique() == 3

assert not structural_period.duplicated(
    ["state_fips", "period"]
).any()

# Modeling features
assert (
    structural_period[STRUCTURAL_FEATURES]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    structural_period[STRUCTURAL_FEATURES]
).all().all()

assert (
    growth_period[GROWTH_FEATURES]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    growth_period[GROWTH_FEATURES]
).all().all()

print("All dataset validations passed.")
print("The data are ready for unsupervised learning.")

In [ ]:
# ============================================================
# ILLINOIS CHECK
# ============================================================

illinois_structural = structural_period.loc[
    structural_period["state"].eq("Illinois")
].copy()

illinois_structural["period"] = pd.Categorical(
    illinois_structural["period"],
    categories=PERIOD_ORDER,
    ordered=True
)

illinois_structural = (
    illinois_structural
    .sort_values("period")
)

display(
    illinois_structural[
        ["state", "period"] + STRUCTURAL_FEATURES
    ]
)


### Step 5A interpretation

Four processed datasets were successfully loaded. The structural panel
contains 500 state-year observations, while the period-level structural
dataset contains 150 observations representing 50 states across three
economic periods.

The eight structural features are complete and contain no missing or
infinite values. The period-level structural dataset will be the primary
input for PCA and clustering.

The growth datasets will not initially be used to determine the clusters.
They will be used after clustering to interpret how the economic performance
of each state and cluster changed during and after the 2020–2022 shock.

# Step 5B -- Exploratory Data Analysis

In [ ]:
# ============================================================
# STEP 5B-1 — ORDER THE RESEARCH PERIODS
# ============================================================

for dataframe in [structural_period, growth_period]:
    dataframe["period"] = pd.Categorical(
        dataframe["period"],
        categories=PERIOD_ORDER,
        ordered=True
    )

structural_period = (
    structural_period
    .sort_values(["period", "state_fips"])
    .reset_index(drop=True)
)

growth_period = (
    growth_period
    .sort_values(["period", "state_fips"])
    .reset_index(drop=True)
)

print(
    structural_period["period"]
    .value_counts(sort=False)
)

### Interpretation: 

Each research period contains exactly 50 observations—one averaged
observation for each state. Therefore, no period is overrepresented in
the clustering analysis.

In [ ]:
# ============================================================
# STEP 5B-2 — DESCRIPTIVE STATISTICS
# ============================================================

structural_summary = (
    structural_period[STRUCTURAL_FEATURES]
    .describe()
    .T
)

structural_summary["skewness"] = (
    structural_period[STRUCTURAL_FEATURES]
    .skew()
)

structural_summary["range"] = (
    structural_summary["max"]
    - structural_summary["min"]
)

display(structural_summary)

In [ ]:
## Period-level means 

period_feature_means = (
    structural_period
    .groupby("period", observed=True)[STRUCTURAL_FEATURES]
    .mean()
    .T
)

display(period_feature_means)

### Structural feature summary

The descriptive statistics show the center, dispersion, range, and
skewness of each structural feature across the 150 state-period
observations.

The monetary and productivity variables are measured in dollars, while
the industry variables are percentages. These differences confirm that
feature standardization will be required before PCA and clustering.

The period means provide an initial description of how the average state
changed between the baseline, shock, and post-shock periods. These
comparisons are descriptive and do not establish causal effects.

In [ ]:
# ============================================================
# STEP 5B-3 — SKEWNESS SCREEN
# ============================================================

skewness_check = (
    structural_period[STRUCTURAL_FEATURES]
    .skew()
    .rename("Skewness")
    .to_frame()
)

skewness_check["Absolute_Skewness"] = (
    skewness_check["Skewness"].abs()
)

skewness_check["Status"] = np.select(
    [
        skewness_check["Absolute_Skewness"] < 0.5,
        skewness_check["Absolute_Skewness"] < 1.0
    ],
    [
        "Low",
        "Moderate"
    ],
    default="High — review transformation"
)

skewness_check = skewness_check.sort_values(
    "Absolute_Skewness",
    ascending=False
)

display(skewness_check)

#### Interpretation guide: 

| Absolute skewness | Interpretation                       |
| ----------------: | ------------------------------------ |
|        Below 0.50 | Low skewness                         |
|         0.50–1.00 | Moderate skewness                    |
|        Above 1.00 | High skewness; review transformation |


In [ ]:
# ============================================================
# STEP 5B-4 — FEATURE DISTRIBUTIONS
# ============================================================

fig, axes = plt.subplots(
    2,
    4,
    figsize=(20, 9)
)

axes = axes.flatten()

for axis, feature in zip(
    axes,
    STRUCTURAL_FEATURES
):
    sns.histplot(
        data=structural_period,
        x=feature,
        kde=True,
        bins=18,
        color="steelblue",
        ax=axis
    )

    axis.set_title(
        feature.replace("_", " ").title(),
        fontsize=11
    )

    axis.set_xlabel("")
    axis.set_ylabel("Count")

plt.suptitle(
    "Distributions of State Structural Features",
    fontsize=16,
    y=1.02
)

plt.tight_layout()
plt.show()

### Feature distributions

The distribution plots help identify asymmetry, unusually concentrated
features, and extreme observations.

**Highly skewed** features may affect **Euclidean distances in K-Means** even
after standardization. Any transformation decision will therefore be
based on both the skewness statistics and the economic meaning of the
feature. 

However, unusual observations will not be removed automatically because
they may represent genuine economic structures, such as resource-intensive
or manufacturing-intensive states.


In [ ]:
# ============================================================
# STEP 5B-5 — PERIOD BOXPLOTS
# ============================================================

fig, axes = plt.subplots(
    2,
    4,
    figsize=(22, 10)
)

axes = axes.flatten()

for axis, feature in zip(
    axes,
    STRUCTURAL_FEATURES
):
    sns.boxplot(
        data=structural_period,
        x="period",
        y=feature,
        order=PERIOD_ORDER,
        color="lightsteelblue",
        ax=axis
    )

    axis.set_title(
        feature.replace("_", " ").title(),
        fontsize=11
    )

    axis.set_xlabel("")
    axis.set_ylabel("")

    axis.tick_params(
        axis="x",
        rotation=20
    )

plt.suptitle(
    "Structural Features Before, During and After the Shock",
    fontsize=16,
    y=1.02
)

plt.tight_layout()
plt.show()

### Period comparison

The boxplots compare the cross-state distributions of each structural
feature across the baseline, shock, and post-shock periods.

A shift in a period's median represents a broad change across states.
Changes in the box width indicate changes in interstate dispersion.
Individual points beyond the whiskers may represent economically distinctive
states rather than data errors.

In [ ]:
# ============================================================
# STEP 5B-6 — CORRELATION MATRIX
# ============================================================

structural_correlation = (
    structural_period[STRUCTURAL_FEATURES]
    .corr()
)

mask = np.triu(
    np.ones_like(
        structural_correlation,
        dtype=bool
    )
)

plt.figure(figsize=(11, 8))

sns.heatmap(
    structural_correlation,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5
)

plt.title(
    "Correlation Between Structural Features",
    fontsize=15
)

plt.tight_layout()
plt.show()

### Identify strong correlations 

In [ ]:
strong_correlations = []

for first_index, first_feature in enumerate(
    STRUCTURAL_FEATURES
):
    for second_feature in STRUCTURAL_FEATURES[
        first_index + 1:
    ]:
        correlation = structural_correlation.loc[
            first_feature,
            second_feature
        ]

        if abs(correlation) >= 0.75:
            strong_correlations.append({
                "Feature_1": first_feature,
                "Feature_2": second_feature,
                "Correlation": correlation
            })

strong_correlations = pd.DataFrame(
    strong_correlations
)

if strong_correlations.empty:
    print("No correlations at or above |0.75|.")
else:
    strong_correlations["Absolute_Correlation"] = (
        strong_correlations["Correlation"].abs()
    )

    strong_correlations = (
        strong_correlations
        .sort_values(
            "Absolute_Correlation",
            ascending=False
        )
        .drop(columns="Absolute_Correlation")
    )

    display(strong_correlations)

### Correlation analysis

Strong correlations are expected among GDP per capita, income, wages,
and output per job because these variables represent related dimensions
of economic capacity.

Correlation does not establish causality. In this project, the correlation
matrix is used to identify overlapping information and to support the PCA
interpretation.

PCA can combine correlated variables into a smaller number of economic
dimensions before clustering.

In [ ]:
# ============================================================
# STEP 5B-7 — IQR OUTLIER SCREEN
# ============================================================

outlier_records = []

for period in PERIOD_ORDER:

    period_data = structural_period.loc[
        structural_period["period"].eq(period)
    ]

    for feature in STRUCTURAL_FEATURES:

        q1 = period_data[feature].quantile(0.25)
        q3 = period_data[feature].quantile(0.75)

        iqr = q3 - q1

        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr

        feature_outliers = period_data.loc[
            ~period_data[feature].between(
                lower_limit,
                upper_limit
            ),
            ["state_fips", "state", "period", feature]
        ].copy()

        if not feature_outliers.empty:
            feature_outliers["feature"] = feature
            feature_outliers["lower_limit"] = lower_limit
            feature_outliers["upper_limit"] = upper_limit

            feature_outliers = feature_outliers.rename(
                columns={feature: "value"}
            )

            outlier_records.append(feature_outliers)

#### Combine the results

In [ ]:
if outlier_records:
    structural_outliers = pd.concat(
        outlier_records,
        ignore_index=True
    )
else:
    structural_outliers = pd.DataFrame(
        columns=[
            "state_fips",
            "state",
            "period",
            "value",
            "feature",
            "lower_limit",
            "upper_limit"
        ]
    )

outlier_summary = (
    structural_outliers
    .groupby(
        ["period", "feature"],
        observed=True
    )
    .size()
    .rename("Outlier_Count")
    .reset_index()
    .sort_values(
        ["period", "Outlier_Count"],
        ascending=[True, False]
    )
)

display(outlier_summary)
display(structural_outliers.head(20))

### Potential outliers

The IQR screen identifies observations that are statistically unusual
within each period. These observations are not automatically considered
errors.

States such as Alaska, Wyoming, North Dakota, or states with specialized
industry structures may legitimately appear as outliers. Removing them
could eliminate economically meaningful information from the clustering
analysis.

Potential outliers will instead be considered when selecting transformations
and when interpreting cluster membership.

In [ ]:
# ============================================================
# STEP 5B-8 — GROWTH FEATURE SUMMARY _ Examine economic growth by period
# ============================================================

growth_period_summary = (
    growth_period
    .groupby("period", observed=True)[GROWTH_FEATURES]
    .agg(["mean", "median", "std"])
)

display(growth_period_summary)

#### Heatmap of average growth

In [ ]:
average_growth_by_period = (
    growth_period
    .groupby("period", observed=True)[GROWTH_FEATURES]
    .mean()
    .reindex(PERIOD_ORDER)
)

plt.figure(figsize=(10, 4))

sns.heatmap(
    average_growth_by_period,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5
)

plt.title(
    "Average State Economic Growth by Period",
    fontsize=14
)

plt.xlabel("Growth feature")
plt.ylabel("Research period")

plt.tight_layout()
plt.show()

### Interpretation: 

The growth indicators provide supporting evidence about economic movement
during each period. They are not initially used to create the structural
clusters.

Negative or weaker average growth during 2020–2022 would indicate the
economic disruption associated with the shock period. Post-shock growth
helps show whether states recovered at similar or different rates.

These results describe economic patterns but do not establish that the
pandemic caused every observed change.

In [ ]:
# ============================================================
# STEP 5B-9 — SAVE EDA OUTPUTS
# ============================================================

RESULTS_DIR = Path("results")
TABLES_DIR = RESULTS_DIR / "tables"

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

structural_summary.to_csv(
    TABLES_DIR /
    "structural_feature_summary.csv"
)

period_feature_means.to_csv(
    TABLES_DIR /
    "structural_period_means.csv"
)

skewness_check.to_csv(
    TABLES_DIR /
    "structural_feature_skewness.csv"
)

structural_correlation.to_csv(
    TABLES_DIR /
    "structural_feature_correlations.csv"
)

outlier_summary.to_csv(
    TABLES_DIR /
    "structural_outlier_summary.csv",
    index=False
)

structural_outliers.to_csv(
    TABLES_DIR /
    "structural_outlier_details.csv",
    index=False
)

average_growth_by_period.to_csv(
    TABLES_DIR /
    "average_growth_by_period.csv"
)

print(f"EDA tables saved in: {TABLES_DIR}")

# Step 5C — Transformation Decisions and Feature Standardization

## 5C.1 Document the transformation decisions

## Step 5C — Transformation Decisions and Feature Standardization

Based on the Step 5B exploratory analysis:

1. `natural_resources_agriculture_share` is strongly right-skewed and will be
   transformed using `log1p`.
2. The remaining features have low or moderate skewness and will remain in their
   original form.
3. Observed outliers will not be automatically removed or winsorized because they
   may represent meaningful structural differences among states.
4. All 12 analytical features will be standardized using `StandardScaler`.
5. A single scaler will be fitted across the complete analytical sample so that
   differences between the baseline, shock, and post-shock periods are preserve.,

## 5C.2 Combine the 12 analytical features

In [ ]:
# ============================================================
# STEP 5C.2 — DEFINE THE FINAL 12 FEATURES
# ============================================================

final_features = STRUCTURAL_FEATURES + GROWTH_FEATURES

print(f"Number of structural features: {len(STRUCTURAL_FEATURES)}")
print(f"Number of growth features: {len(GROWTH_FEATURES)}")
print(f"Total number of final features: {len(final_features)}")

display(
    pd.DataFrame({
        "Feature": final_features,
        "Feature_Type": (
            ["Structural"] * len(STRUCTURAL_FEATURES)
            + ["Growth"] * len(GROWTH_FEATURES)
        )
    })
)

In [ ]:
# ============================================================
# STEP 5C.3 — VALIDATE STRUCTURAL MODELING DATA
# ============================================================

IDENTIFIER_COLUMNS = [
    "state_fips",
    "state",
    "period"
]

required_structural_columns = (
    IDENTIFIER_COLUMNS
    + STRUCTURAL_FEATURES
)

missing_structural_columns = [
    column
    for column in required_structural_columns
    if column not in structural_period.columns
]

if missing_structural_columns:
    raise KeyError(
        "Missing columns in structural_period: "
        f"{missing_structural_columns}"
    )

structural_missing_summary = (
    structural_period[STRUCTURAL_FEATURES]
    .isna()
    .sum()
    .rename("Missing_Count")
    .to_frame()
)

structural_missing_summary["Missing_Percent"] = (
    structural_missing_summary["Missing_Count"]
    / len(structural_period)
    * 100
).round(2)

display(structural_missing_summary)

assert structural_period.shape[0] == 150
assert structural_period["state_fips"].nunique() == 50
assert structural_period["period"].nunique() == 3

assert not structural_period.duplicated(
    ["state_fips", "period"]
).any()

assert (
    structural_period[STRUCTURAL_FEATURES]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    structural_period[STRUCTURAL_FEATURES]
).all().all()

print("Structural modeling data validation passed.")
print("Observations:", len(structural_period))
print("States:", structural_period["state_fips"].nunique())
print("Periods:", structural_period["period"].nunique())
print("Structural features:", len(STRUCTURAL_FEATURES))

In [ ]:
# ============================================================
# STEP 5C.4 — TRANSFORM THE HIGHLY SKEWED FEATURE
# ============================================================

RESOURCE_FEATURE = (
    "natural_resources_agriculture_share"
)

TRANSFORMED_RESOURCE_FEATURE = (
    "log1p_natural_resources_agriculture_share"
)

structural_period_transformed = (
    structural_period.copy()
)

# log1p requires values greater than -1
if (
    structural_period_transformed[RESOURCE_FEATURE]
    <= -1
).any():
    raise ValueError(
        f"{RESOURCE_FEATURE} contains values less than "
        "or equal to -1. log1p cannot be applied."
    )

structural_period_transformed[
    TRANSFORMED_RESOURCE_FEATURE
] = np.log1p(
    structural_period_transformed[
        RESOURCE_FEATURE
    ]
)

# Replace the original resource feature in the modeling list
STRUCTURAL_MODEL_FEATURES = [
    (
        TRANSFORMED_RESOURCE_FEATURE
        if feature == RESOURCE_FEATURE
        else feature
    )
    for feature in STRUCTURAL_FEATURES
]

print(
    "Original structural features:",
    len(STRUCTURAL_FEATURES)
)

print(
    "Transformed modeling features:",
    len(STRUCTURAL_MODEL_FEATURES)
)

display(
    pd.DataFrame({
        "Original_Feature": STRUCTURAL_FEATURES,
        "Modeling_Feature": STRUCTURAL_MODEL_FEATURES
    })
)

In [ ]:
# ============================================================
# STEP 5C.5 — COMPARE SKEWNESS BEFORE AND AFTER TRANSFORMATION
# ============================================================

transformation_comparison = pd.DataFrame({
    "Version": [
        "Original",
        "Log1p transformed"
    ],
    "Feature": [
        RESOURCE_FEATURE,
        TRANSFORMED_RESOURCE_FEATURE
    ],
    "Skewness": [
        structural_period_transformed[
            RESOURCE_FEATURE
        ].skew(),

        structural_period_transformed[
            TRANSFORMED_RESOURCE_FEATURE
        ].skew()
    ]
})

transformation_comparison[
    "Absolute_Skewness"
] = transformation_comparison[
    "Skewness"
].abs()

display(
    transformation_comparison.round(4)
)

In [ ]:
# ============================================================
# STEP 5C.5 — VISUALIZE THE TRANSFORMATION
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

sns.histplot(
    data=structural_period_transformed,
    x=RESOURCE_FEATURE,
    bins=18,
    kde=True,
    color="steelblue",
    ax=axes[0]
)

axes[0].set_title(
    "Original Resource and Agriculture Share"
)
axes[0].set_xlabel(
    "Share (%)"
)
axes[0].set_ylabel(
    "State-period observations"
)

sns.histplot(
    data=structural_period_transformed,
    x=TRANSFORMED_RESOURCE_FEATURE,
    bins=18,
    kde=True,
    color="darkorange",
    ax=axes[1]
)

axes[1].set_title(
    "Log1p-Transformed Resource and Agriculture Share"
)
axes[1].set_xlabel(
    "log1p(share)"
)
axes[1].set_ylabel(
    "State-period observations"
)

plt.suptitle(
    "Effect of the Resource-Share Transformation",
    fontsize=14,
    y=1.03
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5C.6 — STANDARDIZE STRUCTURAL MODELING FEATURES
# ============================================================

structural_scaler = StandardScaler()

X_structural_scaled_array = (
    structural_scaler.fit_transform(
        structural_period_transformed[
            STRUCTURAL_MODEL_FEATURES
        ]
    )
)

X_structural_scaled = pd.DataFrame(
    X_structural_scaled_array,
    columns=STRUCTURAL_MODEL_FEATURES,
    index=structural_period_transformed.index
)

print(
    "Scaled structural matrix shape:",
    X_structural_scaled.shape
)

display(
    X_structural_scaled.head()
)

In [ ]:
# ============================================================
# STEP 5C.7 — VERIFY STANDARDIZATION
# ============================================================

standardization_check = pd.DataFrame({
    "Scaled_Mean": (
        X_structural_scaled.mean()
    ),
    "Scaled_Standard_Deviation": (
        X_structural_scaled.std(ddof=0)
    ),
    "Scaled_Minimum": (
        X_structural_scaled.min()
    ),
    "Scaled_Maximum": (
        X_structural_scaled.max()
    )
})

display(
    standardization_check.round(4)
)

mean_check = np.allclose(
    X_structural_scaled.mean().values,
    0,
    atol=1e-10
)

standard_deviation_check = np.allclose(
    X_structural_scaled.std(ddof=0).values,
    1,
    atol=1e-10
)

print("All scaled means approximately zero:", mean_check)

print(
    "All scaled standard deviations approximately one:",
    standard_deviation_check
)

In [ ]:
# ============================================================
# STEP 5C.8 — CREATE STANDARDIZED STRUCTURAL DATASET
# ============================================================

standardized_structural_period = pd.concat(
    [
        structural_period_transformed[
            IDENTIFIER_COLUMNS
        ].reset_index(drop=True),

        X_structural_scaled.reset_index(
            drop=True
        )
    ],
    axis=1
)

assert standardized_structural_period.shape[0] == 150

assert not standardized_structural_period.duplicated(
    ["state_fips", "period"]
).any()

assert (
    standardized_structural_period[
        STRUCTURAL_MODEL_FEATURES
    ]
    .notna()
    .all()
    .all()
)

print(
    "Standardized structural dataset shape:",
    standardized_structural_period.shape
)

display(
    standardized_structural_period.head()
)

In [ ]:
# ============================================================
# STEP 5C.9 — RECORD PREPROCESSING PARAMETERS
# ============================================================

scaler_parameters = pd.DataFrame({
    "Feature": STRUCTURAL_MODEL_FEATURES,
    "Pre_Scaling_Mean": structural_scaler.mean_,
    "Pre_Scaling_Standard_Deviation": (
        structural_scaler.scale_
    )
})

transformation_decisions = pd.DataFrame({
    "Original_Feature": STRUCTURAL_FEATURES,
    "Modeling_Feature": STRUCTURAL_MODEL_FEATURES,
    "Transformation": [
        (
            "log1p"
            if feature == RESOURCE_FEATURE
            else "None"
        )
        for feature in STRUCTURAL_FEATURES
    ],
    "Standardized": True
})

display(transformation_decisions)
display(scaler_parameters.round(4))

In [ ]:
# ============================================================
# STEP 5C.10 — SAVE PREPROCESSING OUTPUTS
# ============================================================

standardized_structural_period.to_csv(
    PROCESSED_DIR
    / "state_structural_period_standardized.csv",
    index=False
)

structural_period_transformed[
    IDENTIFIER_COLUMNS
    + STRUCTURAL_MODEL_FEATURES
].to_csv(
    PROCESSED_DIR
    / "state_structural_period_transformed.csv",
    index=False
)

transformation_decisions.to_csv(
    TABLES_DIR
    / "structural_transformation_decisions.csv",
    index=False
)

scaler_parameters.to_csv(
    TABLES_DIR
    / "structural_scaler_parameters.csv",
    index=False
)

print("Saved standardized structural dataset:")
print(
    PROCESSED_DIR
    / "state_structural_period_standardized.csv"
)

print("\nSaved transformed structural dataset:")
print(
    PROCESSED_DIR
    / "state_structural_period_transformed.csv"
)

print("\nSaved preprocessing documentation in:")
print(TABLES_DIR)

# Step 5D — Principal Component Analysis

PCA is fitted to the eight standardized structural features across all 150
state-period observations. Using one PCA model provides a common coordinate
system for comparing the baseline, shock, and post-shock periods.

The initial PCA model retains all eight components. The final number of
components will be selected using cumulative explained variance, with an
85% threshold and a minimum of two components.

In [ ]:
# ============================================================
# STEP 5D.1 — FIT THE FULL PCA MODEL
# ============================================================

assert X_structural_scaled.shape == (
    len(structural_period),
    len(STRUCTURAL_MODEL_FEATURES)
)

assert np.isfinite(
    X_structural_scaled
).all().all()

pca_full = PCA()

X_pca_full_array = pca_full.fit_transform(
    X_structural_scaled
)

FULL_COMPONENT_NAMES = [
    f"PC{component_number}"
    for component_number in range(
        1,
        pca_full.n_components_ + 1
    )
]

explained_variance_table = pd.DataFrame({
    "Component": FULL_COMPONENT_NAMES,
    "Eigenvalue": pca_full.explained_variance_,
    "Explained_Variance_Ratio": (
        pca_full.explained_variance_ratio_
    ),
    "Cumulative_Explained_Variance": (
        np.cumsum(
            pca_full.explained_variance_ratio_
        )
    )
})

display(
    explained_variance_table.round(4)
)

print(
    "Total explained variance:",
    explained_variance_table[
        "Explained_Variance_Ratio"
    ].sum().round(4)
)

In [ ]:
# ============================================================
# STEP 5D.2 — EXPLAINED VARIANCE AND SCREE PLOT
# ============================================================

VARIANCE_THRESHOLD = 0.85

fig, variance_axis = plt.subplots(
    figsize=(10, 6)
)

variance_axis.bar(
    explained_variance_table["Component"],
    explained_variance_table[
        "Explained_Variance_Ratio"
    ],
    color="steelblue",
    alpha=0.8,
    label="Individual explained variance"
)

variance_axis.set_xlabel(
    "Principal component"
)
variance_axis.set_ylabel(
    "Individual explained variance ratio",
    color="steelblue"
)
variance_axis.tick_params(
    axis="y",
    labelcolor="steelblue"
)

cumulative_axis = variance_axis.twinx()

cumulative_axis.plot(
    explained_variance_table["Component"],
    explained_variance_table[
        "Cumulative_Explained_Variance"
    ],
    color="darkorange",
    marker="o",
    linewidth=2,
    label="Cumulative explained variance"
)

cumulative_axis.axhline(
    y=VARIANCE_THRESHOLD,
    color="firebrick",
    linestyle="--",
    linewidth=1.5,
    label="85% threshold"
)

cumulative_axis.set_ylabel(
    "Cumulative explained variance",
    color="darkorange"
)
cumulative_axis.tick_params(
    axis="y",
    labelcolor="darkorange"
)
cumulative_axis.set_ylim(0, 1.05)

lines_1, labels_1 = (
    variance_axis.get_legend_handles_labels()
)
lines_2, labels_2 = (
    cumulative_axis.get_legend_handles_labels()
)

variance_axis.legend(
    lines_1 + lines_2,
    labels_1 + labels_2,
    loc="center right"
)

plt.title(
    "PCA Explained Variance for State Economic Structure",
    fontsize=14
)

fig.tight_layout()
plt.show()

| Component | Variance | Cumulative | Suggested interpretation                                  |
| --------- | -------: | ---------: | --------------------------------------------------------- |
| PC1       |   49.68% |     49.68% | Economic capacity and productivity                        |
| PC2       |   19.20% |     68.88% | Resource orientation versus business/industrial structure |
| PC3       |   14.24% |     83.12% | Healthcare and manufacturing concentration                |
| PC4       |    8.70% |     91.82% | Manufacturing versus healthcare/service orientation       |


### Interpret the chart as follows:

- The bars show the information explained by each individual component.
- The orange line shows cumulative information retained.
- The dashed line marks the 85% selection threshold.
- An elbow indicates where additional components begin providing relatively little information.


In [ ]:
# ============================================================
# STEP 5D.3 — SELECT THE NUMBER OF COMPONENTS
# ============================================================

components_for_threshold = (
    np.searchsorted(
        explained_variance_table[
            "Cumulative_Explained_Variance"
        ].values,
        VARIANCE_THRESHOLD
    )
    + 1
)

# Keep at least two dimensions for structural comparison
SELECTED_N_COMPONENTS = max(
    2,
    components_for_threshold
)

selected_cumulative_variance = (
    explained_variance_table.loc[
        SELECTED_N_COMPONENTS - 1,
        "Cumulative_Explained_Variance"
    ]
)

print(
    "Variance threshold:",
    f"{VARIANCE_THRESHOLD:.0%}"
)

print(
    "Components needed to reach threshold:",
    components_for_threshold
)

print(
    "Final number of selected components:",
    SELECTED_N_COMPONENTS
)

print(
    "Cumulative variance retained:",
    f"{selected_cumulative_variance:.2%}"
)

In [ ]:
# ============================================================
# STEP 5D.4 — FIT THE SELECTED PCA MODEL
# ============================================================

pca_model = PCA(
    n_components=SELECTED_N_COMPONENTS
)

X_pca_array = pca_model.fit_transform(
    X_structural_scaled
)

PCA_COMPONENT_NAMES = [
    f"PC{component_number}"
    for component_number in range(
        1,
        SELECTED_N_COMPONENTS + 1
    )
]

X_pca = pd.DataFrame(
    X_pca_array,
    columns=PCA_COMPONENT_NAMES,
    index=X_structural_scaled.index
)

print("Original feature matrix:", X_structural_scaled.shape)
print("Reduced PCA matrix:", X_pca.shape)

display(X_pca.head())

In [ ]:
# ============================================================
# STEP 5D.5 — CALCULATE PCA LOADINGS
# ============================================================

pca_loadings = pd.DataFrame(
    pca_model.components_.T,
    index=STRUCTURAL_MODEL_FEATURES,
    columns=PCA_COMPONENT_NAMES
)

display(
    pca_loadings.round(4)
)

In [ ]:
# ============================================================
# STEP 5D.5 — VISUALIZE PCA LOADINGS
# ============================================================

plt.figure(
    figsize=(
        max(8, SELECTED_N_COMPONENTS * 1.6),
        7
    )
)

sns.heatmap(
    pca_loadings,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={
        "label": "PCA weight"
    }
)

plt.title(
    "Structural Feature Weights in Each Principal Component",
    fontsize=14
)
plt.xlabel("Principal component")
plt.ylabel("Structural feature")

plt.tight_layout()
plt.show()

A large positive or negative value means the feature strongly defines that component. The sign shows direction, but PCA signs are mathematically reversible. Focus primarily on the magnitude and which features move together.


In [ ]:
# ============================================================
# STEP 5D.6 — IDENTIFY DOMINANT FEATURES BY COMPONENT
# ============================================================

dominant_loading_records = []

for component in PCA_COMPONENT_NAMES:

    dominant_features = (
        pca_loadings[component]
        .abs()
        .nlargest(3)
        .index
    )

    for rank, feature in enumerate(
        dominant_features,
        start=1
    ):
        dominant_loading_records.append({
            "Component": component,
            "Rank": rank,
            "Feature": feature,
            "Loading": pca_loadings.loc[
                feature,
                component
            ],
            "Absolute_Loading": abs(
                pca_loadings.loc[
                    feature,
                    component
                ]
            )
        })

dominant_loadings = pd.DataFrame(
    dominant_loading_records
)

display(
    dominant_loadings.round(4)
)

In [ ]:
# ============================================================
# STEP 5D.7 — CREATE STATE-PERIOD PCA DATASET
# ============================================================

pca_scores = pd.concat(
    [
        standardized_structural_period[
            IDENTIFIER_COLUMNS
        ].reset_index(drop=True),

        X_pca.reset_index(drop=True)
    ],
    axis=1
)

assert pca_scores.shape[0] == 150

assert not pca_scores.duplicated(
    ["state_fips", "period"]
).any()

display(pca_scores.head())

print(
    "PCA score dataset shape:",
    pca_scores.shape
)

In [ ]:
# ============================================================
# STEP 5D.8 — PLOT STATE-PERIOD PCA SCORES
# ============================================================

period_colors = {
    "Baseline_2015_2019": "#4C78A8",
    "Shock_2020_2022": "#E45756",
    "Post_Shock_2023_2024": "#54A24B"
}

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=pca_scores,
    x="PC1",
    y="PC2",
    hue="period",
    hue_order=PERIOD_ORDER,
    palette=period_colors,
    s=75,
    alpha=0.75
)

# Calculate period centers in PCA space
period_centroids = (
    pca_scores
    .groupby(
        "period",
        observed=True
    )[["PC1", "PC2"]]
    .mean()
    .reindex(PERIOD_ORDER)
    .reset_index()
)

sns.scatterplot(
    data=period_centroids,
    x="PC1",
    y="PC2",
    hue="period",
    hue_order=PERIOD_ORDER,
    palette=period_colors,
    marker="X",
    s=250,
    edgecolor="black",
    linewidth=1.2,
    legend=False
)

plt.axhline(
    0,
    color="gray",
    linewidth=0.8,
    linestyle="--"
)

plt.axvline(
    0,
    color="gray",
    linewidth=0.8,
    linestyle="--"
)

plt.title(
    "State Economic Structures in PCA Space",
    fontsize=14
)

plt.xlabel(
    f"PC1 ({pca_model.explained_variance_ratio_[0]:.1%} variance)"
)

plt.ylabel(
    f"PC2 ({pca_model.explained_variance_ratio_[1]:.1%} variance)"
)

plt.legend(
    title="Research period",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# OPTIONAL — PAIRWISE PCA VISUALIZATION
# ============================================================

pca_plot_data = pca_scores[
    ["PC1", "PC2", "PC3", "PC4", "period"]
].copy()

sns.pairplot(
    data=pca_plot_data,
    vars=["PC1", "PC2", "PC3", "PC4"],
    hue="period",
    hue_order=PERIOD_ORDER,
    palette=period_colors,
    corner=True,
    plot_kws={
        "s": 45,
        "alpha": 0.7
    },
    diag_kind="hist"
)

plt.suptitle(
    "Pairwise Relationships Among Retained PCA Components",
    fontsize=15,
    y=1.02
)

plt.show()

In [ ]:
# ============================================================
# STEP 5D.9 — EXAMINE ILLINOIS IN PCA SPACE
# ============================================================

illinois_pca = (
    pca_scores.loc[
        pca_scores["state"].eq("Illinois")
    ]
    .copy()
)

illinois_pca["period"] = pd.Categorical(
    illinois_pca["period"],
    categories=PERIOD_ORDER,
    ordered=True
)

illinois_pca = (
    illinois_pca
    .sort_values("period")
)

display(illinois_pca)

In [ ]:
# ============================================================
# VISUALIZE ILLINOIS PCA MOVEMENT
# ============================================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=pca_scores,
    x="PC1",
    y="PC2",
    color="lightgray",
    s=45,
    alpha=0.45
)

plt.plot(
    illinois_pca["PC1"],
    illinois_pca["PC2"],
    color="darkorange",
    marker="o",
    markersize=9,
    linewidth=2
)

for _, row in illinois_pca.iterrows():
    plt.annotate(
        str(row["period"]),
        xy=(row["PC1"], row["PC2"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9
    )

plt.axhline(
    0,
    color="gray",
    linewidth=0.8,
    linestyle="--"
)

plt.axvline(
    0,
    color="gray",
    linewidth=0.8,
    linestyle="--"
)

plt.title(
    "Illinois Structural Movement in PCA Space",
    fontsize=14
)

plt.xlabel(
    f"PC1 ({pca_model.explained_variance_ratio_[0]:.1%} variance)"
)

plt.ylabel(
    f"PC2 ({pca_model.explained_variance_ratio_[1]:.1%} variance)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5D.10 — SAVE PCA OUTPUTS
# ============================================================

explained_variance_table.to_csv(
    TABLES_DIR
    / "pca_explained_variance.csv",
    index=False
)

pca_loadings.to_csv(
    TABLES_DIR
    / "pca_feature_loadings.csv",
    index=True,
    index_label="Feature"
)

dominant_loadings.to_csv(
    TABLES_DIR
    / "pca_dominant_loadings.csv",
    index=False
)

pca_scores.to_csv(
    PROCESSED_DIR
    / "state_structural_pca_scores.csv",
    index=False
)

print("PCA outputs saved successfully.")
print("Selected components:", SELECTED_N_COMPONENTS)
print(
    "Variance retained:",
    f"{selected_cumulative_variance:.2%}"
)

### Step 5D interpretation

Four principal components were retained because they explain **91.82%** of
the total standardized structural variance. *Three components explain
83.12%, which is below the selected 85% threshold*.

- PC1 explains 49.68% of the variance and primarily represents economic
capacity and productivity. It has strong positive weights for real wages,
output per job, GDP per capita, and personal income per capita.

- PC2 explains 19.20% of the variance and contrasts natural-resource and
agricultural orientation with professional-service and manufacturing
orientation.

- PC3 explains 14.24% of the variance and is primarily associated with
healthcare and manufacturing concentration. PC4 explains an additional
8.70% and distinguishes manufacturing-oriented structures from
healthcare- and professional-service-oriented structures.

The PCA score plot shows substantial overlap among the baseline, shock,
and post-shock periods. The period centroids shift slightly toward higher
PC1 values, but interstate structural differences remain much larger than
the average differences between periods.

These patterns are descriptive and do not establish causal effects.
The four retained components provide a compact representation of state
economic structure for the clustering analysis.

# Step 5E — Cluster-Number Selection

K-Means clustering is evaluated using the four retained PCA components,
which preserve 91.82% of the standardized structural variance.

Candidate solutions from two through eight clusters are compared using:

- inertia and the elbow pattern;
- silhouette score;
- Calinski–Harabasz score;
- Davies–Bouldin score;
- cluster-size balance; and
- stability across multiple random seeds.

No single metric will determine the final number of clusters. The selected
solution should be statistically defensible, stable, reasonably balanced,
and economically interpretable.

We will evaluate `k = 2` through `k = 8` using the four retained PCA components.

In [ ]:
# ============================================================
# STEP 5E.1 — VALIDATE THE CLUSTERING INPUT
# ============================================================

EXPECTED_PCA_COLUMNS = [
    f"PC{component_number}"
    for component_number in range(
        1,
        SELECTED_N_COMPONENTS + 1
    )
]

assert list(X_pca.columns) == EXPECTED_PCA_COLUMNS

assert X_pca.shape == (
    150,
    SELECTED_N_COMPONENTS
)

assert X_pca.notna().all().all()
assert np.isfinite(X_pca).all().all()

print("Clustering input validation passed.")
print("Observations:", X_pca.shape[0])
print("PCA components:", X_pca.shape[1])
print("Components used:", list(X_pca.columns))

In [ ]:
# ============================================================
# STEP 5E.2 — EVALUATE CANDIDATE K-MEANS MODELS
# ============================================================

K_VALUES = list(range(2, 9))

candidate_kmeans_models = {}
candidate_cluster_labels = {}

cluster_evaluation_records = []
cluster_size_records = []

for number_of_clusters in K_VALUES:

    candidate_model = KMeans(
        n_clusters=number_of_clusters,
        init="k-means++",
        n_init=50,
        random_state=RANDOM_STATE
    )

    candidate_labels = (
        candidate_model.fit_predict(X_pca)
    )

    candidate_kmeans_models[
        number_of_clusters
    ] = candidate_model

    candidate_cluster_labels[
        number_of_clusters
    ] = candidate_labels

    cluster_counts = (
        pd.Series(candidate_labels)
        .value_counts()
        .sort_index()
    )

    cluster_evaluation_records.append({
        "Number_of_Clusters": number_of_clusters,
        "Inertia": candidate_model.inertia_,
        "Silhouette_Score": silhouette_score(
            X_pca,
            candidate_labels
        ),
        "Calinski_Harabasz_Score":
            calinski_harabasz_score(
                X_pca,
                candidate_labels
            ),
        "Davies_Bouldin_Score":
            davies_bouldin_score(
                X_pca,
                candidate_labels
            ),
        "Smallest_Cluster": cluster_counts.min(),
        "Largest_Cluster": cluster_counts.max(),
        "Smallest_Cluster_Percent": (
            cluster_counts.min()
            / len(candidate_labels)
            * 100
        )
    })

    for cluster_number, cluster_size in (
        cluster_counts.items()
    ):
        cluster_size_records.append({
            "Number_of_Clusters":
                number_of_clusters,
            "Cluster": cluster_number,
            "Observations": cluster_size,
            "Percent": (
                cluster_size
                / len(candidate_labels)
                * 100
            )
        })

cluster_evaluation = pd.DataFrame(
    cluster_evaluation_records
)

cluster_size_details = pd.DataFrame(
    cluster_size_records
)

display(
    cluster_evaluation.round(4)
)

## 5E.3 — Understand the four criteria

| Criterion         | Preferred direction | Meaning                                                                  |
| ----------------- | ------------------: | ------------------------------------------------------------------------ |
| Inertia           |               Lower | Observations are closer to their assigned centroids                      |
| Silhouette        |              Higher | Clusters are compact and well separated                                  |
| Calinski–Harabasz |              Higher | Between-cluster separation is large relative to within-cluster variation |
| Davies–Bouldin    |               Lower | Clusters have less overlap                                               |
| Smallest cluster  |              Review | Detects solutions containing very small clusters                         |


**Inertia** always decreases when more clusters are added. Therefore, we look for an elbow rather than simply choosing the lowest value.

## 5E.4 — Test stability across random seeds

A stable solution should produce similar groupings even when K-Means starts from different initial centroids.

In [ ]:
# ============================================================
# STEP 5E.4 — TEST CLUSTER STABILITY
# ============================================================

from itertools import combinations

STABILITY_SEEDS = [
    7,
    21,
    42,
    84,
    126,
    168,
    210,
    252,
    294,
    336
]

stability_records = []

for number_of_clusters in K_VALUES:

    seed_label_sets = []

    for seed in STABILITY_SEEDS:

        stability_model = KMeans(
            n_clusters=number_of_clusters,
            init="k-means++",
            n_init=20,
            random_state=seed
        )

        seed_labels = (
            stability_model.fit_predict(X_pca)
        )

        seed_label_sets.append(seed_labels)

    pairwise_ari_scores = [
        adjusted_rand_score(
            first_labels,
            second_labels
        )
        for first_labels, second_labels
        in combinations(seed_label_sets, 2)
    ]

    stability_records.append({
        "Number_of_Clusters":
            number_of_clusters,
        "Mean_Stability_ARI":
            np.mean(pairwise_ari_scores),
        "Minimum_Stability_ARI":
            np.min(pairwise_ari_scores),
        "Stability_ARI_Std":
            np.std(pairwise_ari_scores)
    })

cluster_stability = pd.DataFrame(
    stability_records
)

display(
    cluster_stability.round(4)
)

#### Adjusted Rand Index stability is interpreted approximately as:

- Close to 1.00: almost identical cluster assignments
- 0.80–1.00: strong stability
- 0.60–0.80: moderate stability
- Below 0.60: potentially unstable solution


In [ ]:
# ============================================================
# STEP 5E.5 — COMBINE SELECTION RESULTS
# ============================================================

cluster_selection_table = (
    cluster_evaluation
    .merge(
        cluster_stability,
        on="Number_of_Clusters",
        how="left",
        validate="one_to_one"
    )
)

display(
    cluster_selection_table.round(4)
)

In [ ]:
# ============================================================
# STEP 5E.6 — RANK THE CANDIDATE SOLUTIONS
# ============================================================

cluster_selection_ranking = (
    cluster_selection_table.copy()
)

cluster_selection_ranking[
    "Silhouette_Rank"
] = cluster_selection_ranking[
    "Silhouette_Score"
].rank(
    ascending=False,
    method="min"
)

cluster_selection_ranking[
    "Calinski_Harabasz_Rank"
] = cluster_selection_ranking[
    "Calinski_Harabasz_Score"
].rank(
    ascending=False,
    method="min"
)

cluster_selection_ranking[
    "Davies_Bouldin_Rank"
] = cluster_selection_ranking[
    "Davies_Bouldin_Score"
].rank(
    ascending=True,
    method="min"
)

cluster_selection_ranking[
    "Stability_Rank"
] = cluster_selection_ranking[
    "Mean_Stability_ARI"
].rank(
    ascending=False,
    method="min"
)

ranking_columns = [
    "Silhouette_Rank",
    "Calinski_Harabasz_Rank",
    "Davies_Bouldin_Rank",
    "Stability_Rank"
]

cluster_selection_ranking[
    "Mean_Metric_Rank"
] = cluster_selection_ranking[
    ranking_columns
].mean(axis=1)

cluster_selection_ranking = (
    cluster_selection_ranking
    .sort_values(
        [
            "Mean_Metric_Rank",
            "Silhouette_Score"
        ],
        ascending=[True, False]
    )
)

display(
    cluster_selection_ranking[
        [
            "Number_of_Clusters",
            "Silhouette_Score",
            "Calinski_Harabasz_Score",
            "Davies_Bouldin_Score",
            "Mean_Stability_ARI",
            "Minimum_Stability_ARI",
            "Smallest_Cluster",
            "Largest_Cluster",
            "Mean_Metric_Rank"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# STEP 5E.7 — VISUALIZE CLUSTER-SELECTION METRICS
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10)
)

# Inertia
axes[0, 0].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table["Inertia"],
    marker="o",
    color="steelblue",
    linewidth=2
)

axes[0, 0].set_title(
    "Elbow Method"
)
axes[0, 0].set_xlabel(
    "Number of clusters"
)
axes[0, 0].set_ylabel(
    "Inertia"
)

# Silhouette score
axes[0, 1].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table[
        "Silhouette_Score"
    ],
    marker="o",
    color="darkorange",
    linewidth=2
)

axes[0, 1].set_title(
    "Silhouette Score — Higher Is Better"
)
axes[0, 1].set_xlabel(
    "Number of clusters"
)
axes[0, 1].set_ylabel(
    "Silhouette score"
)

# Calinski-Harabasz
axes[1, 0].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table[
        "Calinski_Harabasz_Score"
    ],
    marker="o",
    color="seagreen",
    linewidth=2
)

axes[1, 0].set_title(
    "Calinski–Harabasz Score — Higher Is Better"
)
axes[1, 0].set_xlabel(
    "Number of clusters"
)
axes[1, 0].set_ylabel(
    "Calinski–Harabasz score"
)

# Davies-Bouldin
axes[1, 1].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table[
        "Davies_Bouldin_Score"
    ],
    marker="o",
    color="firebrick",
    linewidth=2
)

axes[1, 1].set_title(
    "Davies–Bouldin Score — Lower Is Better"
)
axes[1, 1].set_xlabel(
    "Number of clusters"
)
axes[1, 1].set_ylabel(
    "Davies–Bouldin score"
)

for axis in axes.flatten():
    axis.set_xticks(K_VALUES)
    axis.grid(
        alpha=0.3
    )

plt.suptitle(
    "K-Means Cluster-Number Evaluation",
    fontsize=16,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5E.8 — VISUALIZE STABILITY AND CLUSTER BALANCE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

axes[0].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table[
        "Mean_Stability_ARI"
    ],
    marker="o",
    linewidth=2,
    color="purple",
    label="Mean ARI"
)

axes[0].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table[
        "Minimum_Stability_ARI"
    ],
    marker="s",
    linestyle="--",
    color="mediumpurple",
    label="Minimum ARI"
)

axes[0].set_title(
    "Cluster Stability Across Random Seeds"
)
axes[0].set_xlabel(
    "Number of clusters"
)
axes[0].set_ylabel(
    "Adjusted Rand Index"
)
axes[0].set_xticks(K_VALUES)
axes[0].set_ylim(0, 1.05)
axes[0].legend()

axes[1].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table[
        "Smallest_Cluster"
    ],
    marker="o",
    linewidth=2,
    color="teal",
    label="Smallest cluster"
)

axes[1].plot(
    cluster_selection_table[
        "Number_of_Clusters"
    ],
    cluster_selection_table[
        "Largest_Cluster"
    ],
    marker="s",
    linewidth=2,
    color="goldenrod",
    label="Largest cluster"
)

axes[1].set_title(
    "Candidate Cluster-Size Range"
)
axes[1].set_xlabel(
    "Number of clusters"
)
axes[1].set_ylabel(
    "State-period observations"
)
axes[1].set_xticks(K_VALUES)
axes[1].legend()

for axis in axes:
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5E.9 — REVIEW CLUSTER-SIZE DETAILS
# ============================================================

cluster_size_pivot = (
    cluster_size_details
    .pivot(
        index="Number_of_Clusters",
        columns="Cluster",
        values="Observations"
    )
    .fillna(0)
    .astype(int)
)

cluster_size_pivot.columns = [
    f"Cluster_{cluster_number}"
    for cluster_number
    in cluster_size_pivot.columns
]

display(cluster_size_pivot)

In [ ]:
# ============================================================
# STEP 5E.10 — SAVE CLUSTER-SELECTION OUTPUTS
# ============================================================

cluster_selection_table.to_csv(
    TABLES_DIR
    / "kmeans_cluster_selection_metrics.csv",
    index=False
)

cluster_selection_ranking.to_csv(
    TABLES_DIR
    / "kmeans_cluster_selection_ranking.csv",
    index=False
)

cluster_size_details.to_csv(
    TABLES_DIR
    / "kmeans_candidate_cluster_sizes.csv",
    index=False
)

print("Cluster-selection outputs saved in:")
print(TABLES_DIR)

In [ ]:
display(
    cluster_selection_table.round(4)
)

display(
    cluster_selection_ranking[
        [
            "Number_of_Clusters",
            "Silhouette_Score",
            "Calinski_Harabasz_Score",
            "Davies_Bouldin_Score",
            "Mean_Stability_ARI",
            "Minimum_Stability_ARI",
            "Smallest_Cluster",
            "Largest_Cluster",
            "Mean_Metric_Rank"
        ]
    ].round(4)
)

display(cluster_size_pivot)

## Why select seven clusters? 

| Consideration         | `k = 2` | `k = 7` | `k = 8` |
| --------------------- | ------: | ------: | ------: |
| Silhouette            |  0.3541 |  0.3076 |  0.3186 |
| Calinski–Harabasz     | 85.2646 | 73.6397 | 70.5250 |
| Davies–Bouldin        |  1.0960 |  1.0626 |  1.0561 |
| Mean stability ARI    |  1.0000 |  0.9576 |  0.7826 |
| Minimum stability ARI |  1.0000 |  0.9138 |  0.6134 |
| Cluster-size range    |  40–110 |   10–28 |   10–26 |


#### The reasons for selecting k = 7 are:
    
- Very strong mean stability: 0.9576
- Strong minimum stability: 0.9138
- Competitive silhouette score: 0.3076
- Competitive Davies–Bouldin score: 1.0626
- Strong Calinski–Harabasz score among the detailed solutions
- Reasonably balanced cluster sizes of 10–28 observations
- Greater economic detail than the two-cluster solution
- Much more stable than the eight-cluster solution

### Step 5E interpretation

The two-cluster solution produced the highest silhouette and
Calinski–Harabasz scores and was perfectly stable across random seeds.
However, it created a broad division containing clusters of 40 and 110
state-period observations. This solution is statistically strong but
provides limited detail for examining different state economic structures.

The seven-cluster solution was selected for the detailed analysis. It
achieved a silhouette score of 0.3076, a Calinski–Harabasz score of
73.6397, and a Davies–Bouldin score of 1.0626. Its mean stability Adjusted
Rand Index was 0.9576, with a minimum of 0.9138, indicating that the
solution was highly consistent across different random initializations.

The seven clusters contain between 10 and 28 state-period observations,
providing a more balanced and informative segmentation than the
two-cluster solution.

The eight-cluster solution produced slightly better separation metrics,
but its mean stability declined to 0.7826 and its minimum stability fell
to 0.6134. Therefore, the additional eighth cluster was not considered
sufficiently reliable.

The moderate silhouette values indicate that **state economic structures
overlap rather than forming perfectly separated natural groups**. The
resulting clusters should therefore be interpreted as descriptive
**economic-structure segments** rather than **definitive classifications**.

| Descriptive segmentation             | Definitive classification             |
| ------------------------------------ | ------------------------------------- |
| Summarizes similar economic patterns | Claims objectively correct categories |
| Boundaries can overlap               | Boundaries are clearly defined        |
| Labels are created after analysis    | Labels are known in advance           |
| Membership may change over time      | Membership is normally fixed          |
| Supports exploration and comparison  | Supports authoritative labeling       |
| No known ground-truth answer         | Ground-truth categories exist         |


In [ ]:
# ============================================================
# STEP 5E.11 — RECORD THE SELECTED CLUSTER COUNT
# ============================================================

SELECTED_K = 7

selected_cluster_metrics = (
    cluster_selection_table.loc[
        cluster_selection_table[
            "Number_of_Clusters"
        ].eq(SELECTED_K)
    ]
    .copy()
    .reset_index(drop=True)
)

assert selected_cluster_metrics.shape[0] == 1

print("Selected number of clusters:", SELECTED_K)

display(
    selected_cluster_metrics.round(4)
)

In [ ]:
# ============================================================
# STEP 5E.12 — SAVE THE CLUSTER-SELECTION DECISION
# ============================================================

selected_cluster_metrics[
    "Selection_Reason"
] = (
    "Strong separation, high stability, balanced sizes, "
    "and useful structural detail"
)

selected_cluster_metrics.to_csv(
    TABLES_DIR
    / "selected_kmeans_cluster_count.csv",
    index=False
)

print(
    "Selected cluster count saved:",
    SELECTED_K
)

# Step 5F — Final K-Means Model and Cluster Profiling

The final K-Means model uses seven clusters and the four retained PCA
components. These components preserve 91.82% of the standardized structural
variance.

The initial cluster numbers are arbitrary model identifiers. Economic names
will be assigned only after examining the PCA centroids, original feature
profiles, standardized feature profiles, and state membership.

Cluster membership is based on all four retained components, even though
some visualizations display only PC1 and PC2.

In [ ]:
# ============================================================
# STEP 5F.1 — FIT THE FINAL K-MEANS MODEL
# ============================================================

assert SELECTED_K == 7

final_kmeans_model = KMeans(
    n_clusters=SELECTED_K,
    init="k-means++",
    n_init=50,
    random_state=RANDOM_STATE
)

final_cluster_labels = (
    final_kmeans_model.fit_predict(X_pca)
)

print("Final number of clusters:", SELECTED_K)
print("Observations clustered:", len(final_cluster_labels))
print("Final model inertia:", round(final_kmeans_model.inertia_, 4))

In [ ]:
# ============================================================
# STEP 5F.2 — ATTACH CLUSTER ASSIGNMENTS
# ============================================================

cluster_assignments = pca_scores.copy()

cluster_assignments["Cluster_ID"] = (
    final_cluster_labels.astype(int)
)

cluster_assignments["Cluster"] = (
    "Cluster_"
    + cluster_assignments[
        "Cluster_ID"
    ].astype(str)
)

assert cluster_assignments.shape[0] == 150

assert not cluster_assignments.duplicated(
    ["state_fips", "period"]
).any()

display(cluster_assignments.head())

In [ ]:
# ============================================================
# STEP 5F.3 — CALCULATE OBSERVATION-LEVEL QUALITY
# ============================================================

from sklearn.metrics import silhouette_samples

cluster_assignments[
    "Silhouette_Value"
] = silhouette_samples(
    X_pca,
    final_cluster_labels
)

distances_to_centroids = (
    final_kmeans_model.transform(X_pca)
)

cluster_assignments[
    "Distance_to_Assigned_Centroid"
] = distances_to_centroids[
    np.arange(len(final_cluster_labels)),
    final_cluster_labels
]

overall_silhouette = silhouette_score(
    X_pca,
    final_cluster_labels
)

print(
    "Overall silhouette score:",
    round(overall_silhouette, 4)
)

In [ ]:
# ============================================================
# STEP 5F.4 — CLUSTER-SPECIFIC QUALITY SUMMARY
# ============================================================

cluster_quality_summary = (
    cluster_assignments
    .groupby("Cluster")
    .agg(
        Observations=(
            "state_fips",
            "size"
        ),
        Mean_Silhouette=(
            "Silhouette_Value",
            "mean"
        ),
        Median_Silhouette=(
            "Silhouette_Value",
            "median"
        ),
        Minimum_Silhouette=(
            "Silhouette_Value",
            "min"
        ),
        Mean_Distance_to_Centroid=(
            "Distance_to_Assigned_Centroid",
            "mean"
        ),
        Maximum_Distance_to_Centroid=(
            "Distance_to_Assigned_Centroid",
            "max"
        )
    )
    .reset_index()
)

cluster_quality_summary[
    "Percent_of_Observations"
] = (
    cluster_quality_summary["Observations"]
    / len(cluster_assignments)
    * 100
)

display(
    cluster_quality_summary.round(4)
)

In [ ]:
# ============================================================
# STEP 5F.5 — CLUSTER SIZES BY PERIOD
# ============================================================

cluster_sizes_by_period = pd.crosstab(
    index=cluster_assignments["Cluster"],
    columns=cluster_assignments["period"]
).reindex(
    columns=PERIOD_ORDER,
    fill_value=0
)

cluster_sizes_by_period[
    "Total"
] = cluster_sizes_by_period.sum(axis=1)

display(cluster_sizes_by_period)

In [ ]:
# ============================================================
# STEP 5F.6 — PCA-SPACE CLUSTER CENTROIDS
# ============================================================

cluster_centroids_pca = pd.DataFrame(
    final_kmeans_model.cluster_centers_,
    columns=PCA_COMPONENT_NAMES
)

cluster_centroids_pca.insert(
    0,
    "Cluster_ID",
    range(SELECTED_K)
)

cluster_centroids_pca.insert(
    1,
    "Cluster",
    (
        "Cluster_"
        + cluster_centroids_pca[
            "Cluster_ID"
        ].astype(str)
    )
)

display(
    cluster_centroids_pca.round(4)
)

Use the PCA interpretations established previously:
- PC1: economic capacity and productivity
- PC2: resource orientation
- PC3: healthcare and manufacturing concentration
- PC4: manufacturing versus healthcare/services

In [ ]:
# ============================================================
# STEP 5F.7 — JOIN CLUSTERS TO ORIGINAL ECONOMIC FEATURES
# ============================================================

assignment_columns = (
    IDENTIFIER_COLUMNS
    + [
        "Cluster_ID",
        "Cluster",
        "Silhouette_Value",
        "Distance_to_Assigned_Centroid"
    ]
)

clustered_structural_period = (
    cluster_assignments[
        assignment_columns
    ]
    .merge(
        structural_period[
            IDENTIFIER_COLUMNS
            + STRUCTURAL_FEATURES
        ],
        on=IDENTIFIER_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

assert clustered_structural_period.shape[0] == 150

assert (
    clustered_structural_period[
        STRUCTURAL_FEATURES
    ]
    .notna()
    .all()
    .all()
)

display(
    clustered_structural_period.head()
)

In [ ]:
# ============================================================
# STEP 5F.8 — ORIGINAL-UNIT CLUSTER PROFILES
# ============================================================

cluster_profile_original = (
    clustered_structural_period
    .groupby("Cluster")[
        STRUCTURAL_FEATURES
    ]
    .mean()
)

display(
    cluster_profile_original.round(2)
)

In [ ]:
# ============================================================
# STEP 5F.9 — STANDARDIZED CLUSTER PROFILES
# ============================================================

clustered_standardized_period = (
    cluster_assignments[
        IDENTIFIER_COLUMNS
        + [
            "Cluster_ID",
            "Cluster"
        ]
    ]
    .merge(
        standardized_structural_period[
            IDENTIFIER_COLUMNS
            + STRUCTURAL_MODEL_FEATURES
        ],
        on=IDENTIFIER_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

cluster_profile_standardized = (
    clustered_standardized_period
    .groupby("Cluster")[
        STRUCTURAL_MODEL_FEATURES
    ]
    .mean()
)

display(
    cluster_profile_standardized.round(3)
)

Interpret standardized means as:
    
- Positive: above the overall state-period average
- Negative: below the overall state-period average
- Near zero: close to the overall average
- Above `+1` or below `−1`: especially distinctive


In [ ]:
# ============================================================
# STEP 5F.10 — CLUSTER PROFILE HEATMAP
# ============================================================

plt.figure(figsize=(14, 7))

sns.heatmap(
    cluster_profile_standardized,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={
        "label": "Mean standardized feature value"
    }
)

plt.title(
    "Standardized Economic Profiles of the Seven Clusters",
    fontsize=15
)
plt.xlabel("Structural feature")
plt.ylabel("Cluster")

plt.yticks(rotation=45)
plt.xticks(
    rotation=40,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5F.11 — IDENTIFY DISTINGUISHING CLUSTER FEATURES
# ============================================================

distinguishing_feature_records = []

for cluster_name in (
    cluster_profile_standardized.index
):

    cluster_values = (
        cluster_profile_standardized.loc[
            cluster_name
        ]
    )

    highest_features = (
        cluster_values
        .nlargest(3)
        .index
    )

    lowest_features = (
        cluster_values
        .nsmallest(3)
        .index
    )

    for rank, feature in enumerate(
        highest_features,
        start=1
    ):
        distinguishing_feature_records.append({
            "Cluster": cluster_name,
            "Direction": "High",
            "Rank": rank,
            "Feature": feature,
            "Standardized_Mean":
                cluster_values[feature]
        })

    for rank, feature in enumerate(
        lowest_features,
        start=1
    ):
        distinguishing_feature_records.append({
            "Cluster": cluster_name,
            "Direction": "Low",
            "Rank": rank,
            "Feature": feature,
            "Standardized_Mean":
                cluster_values[feature]
        })

distinguishing_cluster_features = pd.DataFrame(
    distinguishing_feature_records
)

display(
    distinguishing_cluster_features.round(3)
)

In [ ]:
# ============================================================
# STEP 5F.12 — VISUALIZE FINAL CLUSTERS
# ============================================================

CLUSTER_ORDER = [
    f"Cluster_{cluster_number}"
    for cluster_number in range(SELECTED_K)
]

cluster_colors = dict(
    zip(
        CLUSTER_ORDER,
        sns.color_palette(
            "tab10",
            SELECTED_K
        )
    )
)

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=cluster_assignments,
    x="PC1",
    y="PC2",
    hue="Cluster",
    hue_order=CLUSTER_ORDER,
    palette=cluster_colors,
    s=80,
    alpha=0.75
)

for _, centroid in (
    cluster_centroids_pca.iterrows()
):
    cluster_name = centroid["Cluster"]

    plt.scatter(
        centroid["PC1"],
        centroid["PC2"],
        marker="X",
        s=260,
        color=cluster_colors[cluster_name],
        edgecolor="black",
        linewidth=1.3
    )

    plt.annotate(
        cluster_name,
        xy=(
            centroid["PC1"],
            centroid["PC2"]
        ),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=9,
        weight="bold"
    )

plt.axhline(
    0,
    color="gray",
    linestyle="--",
    linewidth=0.8
)

plt.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=0.8
)

plt.title(
    "Seven State Economic-Structure Clusters in PCA Space",
    fontsize=15
)

plt.xlabel(
    f"PC1 ({pca_model.explained_variance_ratio_[0]:.1%} variance)"
)

plt.ylabel(
    f"PC2 ({pca_model.explained_variance_ratio_[1]:.1%} variance)"
)

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5F.13 — CLUSTERS WITHIN EACH RESEARCH PERIOD
# ============================================================

cluster_period_grid = sns.FacetGrid(
    cluster_assignments,
    col="period",
    col_order=PERIOD_ORDER,
    hue="Cluster",
    hue_order=CLUSTER_ORDER,
    palette=cluster_colors,
    height=4.5,
    aspect=1
)

cluster_period_grid.map_dataframe(
    sns.scatterplot,
    x="PC1",
    y="PC2",
    s=70,
    alpha=0.8
)

cluster_period_grid.add_legend(
    title="Cluster"
)

cluster_period_grid.set_axis_labels(
    "PC1",
    "PC2"
)

cluster_period_grid.set_titles(
    "{col_name}"
)

cluster_period_grid.fig.subplots_adjust(
    top=0.82
)

cluster_period_grid.fig.suptitle(
    "State Economic-Structure Clusters by Period",
    fontsize=15
)

plt.show()

In [ ]:
# ============================================================
# STEP 5F.14 — BASELINE CLUSTER MEMBERSHIP
# ============================================================

baseline_cluster_membership = (
    cluster_assignments.loc[
        cluster_assignments["period"].eq(
            "Baseline_2015_2019"
        ),
        [
            "state_fips",
            "state",
            "Cluster_ID",
            "Cluster"
        ]
    ]
    .sort_values(
        ["Cluster_ID", "state"]
    )
    .reset_index(drop=True)
)

assert baseline_cluster_membership.shape[0] == 50

display(
    baseline_cluster_membership
)

In [ ]:
# ============================================================
# STEP 5F.14 — BASELINE STATES BY CLUSTER
# ============================================================

baseline_cluster_state_lists = (
    baseline_cluster_membership
    .groupby("Cluster")
    .agg(
        Number_of_States=(
            "state",
            "size"
        ),
        States=(
            "state",
            lambda states: ", ".join(
                sorted(states)
            )
        )
    )
    .reset_index()
)

display(
    baseline_cluster_state_lists
)

In [ ]:
# ============================================================
# STEP 5F.15 — SAVE FINAL CLUSTERING OUTPUTS
# ============================================================

cluster_assignments.to_csv(
    PROCESSED_DIR
    / "state_period_cluster_assignments.csv",
    index=False
)

clustered_structural_period.to_csv(
    PROCESSED_DIR
    / "state_period_clusters_with_features.csv",
    index=False
)

cluster_quality_summary.to_csv(
    TABLES_DIR
    / "final_cluster_quality_summary.csv",
    index=False
)

cluster_sizes_by_period.to_csv(
    TABLES_DIR
    / "final_cluster_sizes_by_period.csv"
)

cluster_centroids_pca.to_csv(
    TABLES_DIR
    / "final_cluster_centroids_pca.csv",
    index=False
)

cluster_profile_original.to_csv(
    TABLES_DIR
    / "final_cluster_profiles_original_units.csv"
)

cluster_profile_standardized.to_csv(
    TABLES_DIR
    / "final_cluster_profiles_standardized.csv"
)

distinguishing_cluster_features.to_csv(
    TABLES_DIR
    / "final_cluster_distinguishing_features.csv",
    index=False
)

baseline_cluster_membership.to_csv(
    TABLES_DIR
    / "baseline_cluster_membership.csv",
    index=False
)

baseline_cluster_state_lists.to_csv(
    TABLES_DIR
    / "baseline_states_by_cluster.csv",
    index=False
)

print("Final clustering outputs saved successfully.")

In [ ]:
display(cluster_quality_summary.round(4))
display(cluster_centroids_pca.round(4))
display(cluster_profile_standardized.round(3))
display(baseline_cluster_state_lists)

## Cluster Interpretation 

| Cluster   | Suggested descriptive name                       | Main evidence                                                                                          |
| --------- | ------------------------------------------------ | ------------------------------------------------------------------------------------------------------ |
| Cluster 0 | High-Capacity Professional-Service Economies     | Above-average GDP, income, productivity, and wages; professional services `+1.273`; low resource share |
| Cluster 1 | Lower-Capacity Resource–Healthcare Economies     | Below-average capacity measures; resource share `+0.923`; healthcare `+0.657`                          |
| Cluster 2 | Balanced Manufacturing–Healthcare Economies      | Capacity near the national average; manufacturing `+0.542`; healthcare `+0.534`                        |
| Cluster 3 | High-Capacity Resource-Specialized Economies     | Resource share `+2.208`; GDP `+1.023`; very low professional services and manufacturing                |
| Cluster 4 | Very-High-Capacity Service–Healthcare Economies  | Extremely high GDP, income, productivity, and wages; healthcare `+1.031`                               |
| Cluster 5 | Professional-Service, Low-Healthcare Economies   | Professional services `+0.608`; healthcare `−1.227`; other capacity measures near average              |
| Cluster 6 | Lower-Capacity Manufacturing-Intensive Economies | Manufacturing `+1.417`; below-average GDP, income, productivity, and wages                             |


In [ ]:
# ============================================================
# STEP 5F.16 — ASSIGN DESCRIPTIVE CLUSTER NAMES
# ============================================================

CLUSTER_NAME_MAP = {
    0: "High-Capacity Professional-Service Economies",
    1: "Lower-Capacity Resource-Healthcare Economies",
    2: "Balanced Manufacturing-Healthcare Economies",
    3: "High-Capacity Resource-Specialized Economies",
    4: "Very-High-Capacity Service-Healthcare Economies",
    5: "Professional-Service Low-Healthcare Economies",
    6: "Lower-Capacity Manufacturing-Intensive Economies"
}

cluster_name_table = pd.DataFrame({
    "Cluster_ID": list(CLUSTER_NAME_MAP.keys()),
    "Cluster_Name": list(CLUSTER_NAME_MAP.values())
})

display(cluster_name_table)

In [ ]:
# ============================================================
# STEP 5F.17 — APPLY NAMES TO CLUSTER DATASETS
# ============================================================

cluster_assignments["Cluster_Name"] = (
    cluster_assignments["Cluster_ID"]
    .map(CLUSTER_NAME_MAP)
)

clustered_structural_period["Cluster_Name"] = (
    clustered_structural_period["Cluster_ID"]
    .map(CLUSTER_NAME_MAP)
)

clustered_standardized_period["Cluster_Name"] = (
    clustered_standardized_period["Cluster_ID"]
    .map(CLUSTER_NAME_MAP)
)

baseline_cluster_membership["Cluster_Name"] = (
    baseline_cluster_membership["Cluster_ID"]
    .map(CLUSTER_NAME_MAP)
)

assert cluster_assignments[
    "Cluster_Name"
].notna().all()

print("Descriptive cluster names assigned successfully.")

In [ ]:
# ============================================================
# STEP 5F.18 — COMPLETE BASELINE STATE LISTS
# ============================================================

baseline_cluster_state_lists_labeled = (
    baseline_cluster_membership
    .groupby(
        [
            "Cluster_ID",
            "Cluster_Name"
        ]
    )
    .agg(
        Number_of_States=(
            "state",
            "size"
        ),
        States=(
            "state",
            lambda states: ", ".join(
                sorted(states)
            )
        )
    )
    .reset_index()
    .sort_values("Cluster_ID")
)

with pd.option_context(
    "display.max_colwidth",
    None,
    "display.width",
    None
):
    display(
        baseline_cluster_state_lists_labeled
    )

In [ ]:
# ============================================================
# STEP 5F.19 — REVIEW WASHINGTON'S BASELINE PROFILE
# ============================================================

washington_baseline_profile = (
    clustered_standardized_period.loc[
        clustered_standardized_period[
            "state"
        ].eq("Washington")
        & clustered_standardized_period[
            "period"
        ].eq("Baseline_2015_2019"),
        IDENTIFIER_COLUMNS
        + [
            "Cluster_ID",
            "Cluster_Name"
        ]
        + STRUCTURAL_MODEL_FEATURES
    ]
)

display(
    washington_baseline_profile.round(3)
)

In [ ]:
# ============================================================
# STEP 5F.20 — SAVE LABELED CLUSTER OUTPUTS
# ============================================================

cluster_assignments.to_csv(
    PROCESSED_DIR
    / "state_period_cluster_assignments_labeled.csv",
    index=False
)

clustered_structural_period.to_csv(
    PROCESSED_DIR
    / "state_period_clusters_with_features_labeled.csv",
    index=False
)

cluster_name_table.to_csv(
    TABLES_DIR
    / "cluster_descriptive_names.csv",
    index=False
)

baseline_cluster_state_lists_labeled.to_csv(
    TABLES_DIR
    / "baseline_states_by_named_cluster.csv",
    index=False
)

print("Labeled cluster outputs saved successfully.")

### Step 5F interpretation

The final seven-cluster model produced economically distinguishable state
profiles. Cluster sizes ranged from 10 to 28 state-period observations,
and all clusters had positive mean silhouette values.

Cluster 3 was the most clearly separated group, with a mean silhouette
value of 0.5190. It represents high-capacity, resource-specialized
economies. Cluster 4 was also relatively distinct, with a mean silhouette
of 0.3967, and represents very-high-capacity service- and
healthcare-oriented economies.

Cluster 1 had the weakest separation, with a mean silhouette of 0.2369
and a minimum observation-level value of -0.0521. Some observations in
this group may therefore share characteristics with neighboring clusters.

The standardized profiles show that the clusters differ across economic
capacity, productivity, wages, industry structure, healthcare employment,
professional services, manufacturing, and natural-resource orientation.

These clusters are descriptive economic-structure segments. Their names
summarize relative feature patterns and should not be interpreted as
official or permanent classifications of the states.

In [ ]:
# ============================================================
# STEP 5F.21 — PREPARE BASELINE MAP DATA
# ============================================================

import plotly.express as px
import plotly.graph_objects as go

STATE_ABBREVIATIONS = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY"
}

CLUSTER_MAP_LABELS = {
    0: "0 — High-Capacity Professional Service",
    1: "1 — Lower-Capacity Resource–Healthcare",
    2: "2 — Balanced Manufacturing–Healthcare",
    3: "3 — High-Capacity Resource Specialized",
    4: "4 — Very-High-Capacity Service–Healthcare",
    5: "5 — Professional Service, Low Healthcare",
    6: "6 — Lower-Capacity Manufacturing Intensive"
}

baseline_cluster_map = (
    baseline_cluster_membership.copy()
)

baseline_cluster_map["State_Code"] = (
    baseline_cluster_map["state"]
    .map(STATE_ABBREVIATIONS)
)

baseline_cluster_map["Cluster_Map_Label"] = (
    baseline_cluster_map["Cluster_ID"]
    .map(CLUSTER_MAP_LABELS)
)

baseline_cluster_map["Research_Period"] = (
    "Baseline: 2015–2019"
)

assert baseline_cluster_map.shape[0] == 50

assert baseline_cluster_map[
    "State_Code"
].notna().all()

assert baseline_cluster_map[
    "Cluster_Name"
].notna().all()

display(
    baseline_cluster_map[
        [
            "state",
            "State_Code",
            "Cluster_ID",
            "Cluster_Name"
        ]
    ].head()
)

In [ ]:
# ============================================================
# STEP 5F.22 — DEFINE CONSISTENT CLUSTER COLORS
# ============================================================

CLUSTER_MAP_COLORS = {
    "0 — High-Capacity Professional Service": "#A9C4E4",
    "1 — Lower-Capacity Resource–Healthcare": "#F6BC7B",
    "2 — Balanced Manufacturing–Healthcare": "#A7D6D2",
    "3 — High-Capacity Resource Specialized": "#D0ADD0",
    "4 — Very-High-Capacity Service–Healthcare": "#F2A6A6",
    "5 — Professional Service, Low Healthcare": "#F3DE87",
    "6 — Lower-Capacity Manufacturing Intensive": "#A8D19A"
}

CLUSTER_MAP_ORDER = [
    CLUSTER_MAP_LABELS[cluster_id]
    for cluster_id in range(SELECTED_K)
]

In [ ]:
# ============================================================
# STEP 5F.23 — BASELINE ECONOMIC-STRUCTURE CLUSTER MAP
# ============================================================

baseline_cluster_figure = px.choropleth(
    baseline_cluster_map,
    locations="State_Code",
    locationmode="USA-states",
    scope="usa",
    color="Cluster_Map_Label",
    color_discrete_map=CLUSTER_MAP_COLORS,
    category_orders={
        "Cluster_Map_Label": CLUSTER_MAP_ORDER
    },
    hover_name="state",
    hover_data={
        "State_Code": False,
        "Cluster_ID": True,
        "Cluster_Name": True,
        "Research_Period": True,
        "Cluster_Map_Label": False
    },
    labels={
        "Cluster_ID": "Cluster",
        "Cluster_Name": "Economic profile",
        "Research_Period": "Period",
        "Cluster_Map_Label": "Structural cluster"
    }
)

# Add the cluster number to each state
baseline_cluster_figure.add_trace(
    go.Scattergeo(
        locations=baseline_cluster_map[
            "State_Code"
        ],
        locationmode="USA-states",
        mode="text",
        text=baseline_cluster_map[
            "Cluster_ID"
        ].astype(str),
        textfont={
            "size": 10,
            "color": "#202020"
        },
        hoverinfo="skip",
        showlegend=False
    )
)

# Highlight Illinois
baseline_cluster_figure.add_trace(
    go.Scattergeo(
        lon=[-89.1965],
        lat=[40.0417],
        mode="markers+text",
        marker={
            "size": 15,
            "color": "#111111",
            "symbol": "star",
            "line": {
                "color": "white",
                "width": 1
            }
        },
        text=["Illinois"],
        textposition="top center",
        textfont={
            "size": 12,
            "color": "#111111"
        },
        name="Illinois",
        hovertemplate=(
            "<b>Illinois</b><br>"
            "Cluster 0<br>"
            "High-Capacity Professional-Service Economies"
            "<extra></extra>"
        )
    )
)

baseline_cluster_figure.update_geos(
    showland=True,
    landcolor="#F4F4F4",
    showlakes=True,
    lakecolor="#FFFFFF",
    showsubunits=True,
    subunitcolor="#FFFFFF",
    subunitwidth=1.2
)

baseline_cluster_figure.update_layout(
    title={
        "text": (
            "U.S. State Economic-Structure Clusters"
            "<br>"
            "<sup>"
            "Baseline period: 2015–2019 | "
            "Seven-cluster K-Means model using four PCA components"
            "</sup>"
        ),
        "x": 0.5,
        "xanchor": "center"
    },
    width=1450,
    height=720,
    margin={
        "l": 20,
        "r": 390,
        "t": 100,
        "b": 30
    },
    legend={
        "title": {
            "text": "Economic-structure segment"
        },
        "x": 1.01,
        "y": 1,
        "xanchor": "left",
        "yanchor": "top",
        "font": {
            "size": 11
        }
    },
    paper_bgcolor="white",
    geo_bgcolor="white"
)

baseline_cluster_figure.show()

In [ ]:
# ============================================================
# STEP 5F.24 — SAVE INTERACTIVE BASELINE MAP
# ============================================================

FIGURES_DIR = RESULTS_DIR / "figures"

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

baseline_map_path = (
    FIGURES_DIR
    / "baseline_economic_structure_clusters.html"
)

baseline_cluster_figure.write_html(
    baseline_map_path,
    include_plotlyjs=True
)

baseline_cluster_map.to_csv(
    TABLES_DIR
    / "baseline_cluster_map_data.csv",
    index=False
)

print("Interactive cluster map saved to:")
print(baseline_map_path)

### Baseline cluster geography

The baseline map shows that economic-structure clusters do not follow
simple geographic boundaries. Some neighboring states belong to the same
cluster, particularly within the manufacturing-intensive group, but other
clusters contain states from geographically distant regions.

Illinois belongs to the High-Capacity Professional-Service cluster with
California, Delaware, Maryland, New Jersey, and Virginia.

The Lower-Capacity Manufacturing-Intensive cluster is concentrated mainly
in the Midwest and South. The Very-High-Capacity Service–Healthcare
cluster contains Connecticut, Massachusetts, and New York.

The High-Capacity Resource-Specialized cluster includes Alaska, North
Dakota, Washington, and Wyoming. This group is geographically dispersed,
showing that cluster membership reflects multivariate economic structure
rather than geographic proximity alone.

The map represents descriptive similarities during the 2015–2019
baseline period and does not imply that geography caused the observed
economic profiles.

In [ ]:
# ============================================================
# STEP 5F.25 — PREPARE ALL-PERIOD MAP DATA
# ============================================================

PERIOD_DISPLAY_LABELS = {
    "Baseline_2015_2019": "Baseline: 2015–2019",
    "Shock_2020_2022": "Shock: 2020–2022",
    "Post_Shock_2023_2024": "Post-shock: 2023–2024"
}

all_period_cluster_map = (
    cluster_assignments.copy()
)

# Recreate names if necessary
if "Cluster_Name" not in all_period_cluster_map.columns:
    all_period_cluster_map["Cluster_Name"] = (
        all_period_cluster_map["Cluster_ID"]
        .map(CLUSTER_NAME_MAP)
    )

all_period_cluster_map["State_Code"] = (
    all_period_cluster_map["state"]
    .map(STATE_ABBREVIATIONS)
)

all_period_cluster_map["Period_Label"] = (
    all_period_cluster_map["period"]
    .astype(str)
    .map(PERIOD_DISPLAY_LABELS)
)

all_period_cluster_map["Cluster_Map_Label"] = (
    all_period_cluster_map["Cluster_ID"]
    .map(CLUSTER_MAP_LABELS)
)

assert all_period_cluster_map.shape[0] == 150

assert all_period_cluster_map[
    [
        "State_Code",
        "Period_Label",
        "Cluster_Name",
        "Cluster_Map_Label"
    ]
].notna().all().all()

display(
    all_period_cluster_map[
        [
            "state",
            "Period_Label",
            "Cluster_ID",
            "Cluster_Name"
        ]
    ].head()
)

In [ ]:
# ============================================================
# STEP 5F.26 — POST-SHOCK CLUSTER MAP
# ============================================================

post_shock_map_data = (
    all_period_cluster_map.loc[
        all_period_cluster_map[
            "period"
        ].astype(str).eq(
            "Post_Shock_2023_2024"
        )
    ]
    .copy()
)

assert post_shock_map_data.shape[0] == 50

post_shock_cluster_figure = px.choropleth(
    post_shock_map_data,
    locations="State_Code",
    locationmode="USA-states",
    scope="usa",
    color="Cluster_Map_Label",
    color_discrete_map=CLUSTER_MAP_COLORS,
    category_orders={
        "Cluster_Map_Label": CLUSTER_MAP_ORDER
    },
    hover_name="state",
    hover_data={
        "State_Code": False,
        "Cluster_ID": True,
        "Cluster_Name": True,
        "Period_Label": True,
        "Cluster_Map_Label": False
    },
    labels={
        "Cluster_ID": "Cluster",
        "Cluster_Name": "Economic profile",
        "Period_Label": "Period",
        "Cluster_Map_Label": "Structural cluster"
    }
)

# Display the cluster number in each state
post_shock_cluster_figure.add_trace(
    go.Scattergeo(
        locations=post_shock_map_data[
            "State_Code"
        ],
        locationmode="USA-states",
        mode="text",
        text=post_shock_map_data[
            "Cluster_ID"
        ].astype(str),
        textfont={
            "size": 10,
            "color": "#202020"
        },
        hoverinfo="skip",
        showlegend=False
    )
)

# Highlight Illinois
post_shock_illinois = (
    post_shock_map_data.loc[
        post_shock_map_data["state"].eq(
            "Illinois"
        )
    ]
    .iloc[0]
)

post_shock_cluster_figure.add_trace(
    go.Scattergeo(
        locations=["IL"],
        locationmode="USA-states",
        mode="markers+text",
        marker={
            "size": 15,
            "color": "#111111",
            "symbol": "star",
            "line": {
                "color": "white",
                "width": 1
            }
        },
        text=["Illinois"],
        textposition="top center",
        name="Illinois",
        hovertemplate=(
            "<b>Illinois</b><br>"
            f"Cluster {post_shock_illinois['Cluster_ID']}<br>"
            f"{post_shock_illinois['Cluster_Name']}"
            "<extra></extra>"
        )
    )
)

post_shock_cluster_figure.update_geos(
    showland=True,
    landcolor="#F4F4F4",
    showlakes=True,
    lakecolor="white",
    showsubunits=True,
    subunitcolor="white",
    subunitwidth=1.2
)

post_shock_cluster_figure.update_layout(
    title={
        "text": (
            "U.S. State Economic-Structure Clusters"
            "<br>"
            "<sup>Post-shock period: 2023–2024</sup>"
        ),
        "x": 0.5,
        "xanchor": "center"
    },
    width=1450,
    height=720,
    margin={
        "l": 20,
        "r": 390,
        "t": 100,
        "b": 30
    },
    legend={
        "title": {
            "text": "Economic-structure segment"
        },
        "x": 1.01,
        "y": 1,
        "xanchor": "left",
        "yanchor": "top"
    },
    paper_bgcolor="white",
    geo_bgcolor="white"
)

post_shock_cluster_figure.show()

In [ ]:
# ============================================================
# STEP 5F.27 — THREE-PERIOD CLUSTER MAP COMPARISON
# ============================================================

PERIOD_LABEL_ORDER = [
    PERIOD_DISPLAY_LABELS[period]
    for period in PERIOD_ORDER
]

period_comparison_figure = px.choropleth(
    all_period_cluster_map,
    locations="State_Code",
    locationmode="USA-states",
    scope="usa",
    color="Cluster_Map_Label",
    facet_col="Period_Label",
    facet_col_spacing=0.015,
    color_discrete_map=CLUSTER_MAP_COLORS,
    category_orders={
        "Period_Label": PERIOD_LABEL_ORDER,
        "Cluster_Map_Label": CLUSTER_MAP_ORDER
    },
    hover_name="state",
    hover_data={
        "State_Code": False,
        "Cluster_ID": True,
        "Cluster_Name": True,
        "Period_Label": True,
        "Cluster_Map_Label": False
    },
    labels={
        "Cluster_ID": "Cluster",
        "Cluster_Name": "Economic profile",
        "Period_Label": "Period",
        "Cluster_Map_Label": "Structural cluster"
    }
)

# Simplify the facet titles
period_comparison_figure.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.split("=")[-1]
    )
)

period_comparison_figure.update_geos(
    showland=True,
    landcolor="#F4F4F4",
    showlakes=True,
    lakecolor="white",
    showsubunits=True,
    subunitcolor="white",
    subunitwidth=1
)

period_comparison_figure.update_layout(
    title={
        "text": (
            "Comparison of U.S. State Economic-Structure Clusters"
            "<br>"
            "<sup>"
            "Consistent cluster definitions and colors across all periods"
            "</sup>"
        ),
        "x": 0.5,
        "xanchor": "center"
    },
    width=1900,
    height=650,
    margin={
        "l": 10,
        "r": 380,
        "t": 110,
        "b": 20
    },
    legend={
        "title": {
            "text": "Economic-structure segment"
        },
        "x": 1.01,
        "y": 1
    },
    paper_bgcolor="white"
)

period_comparison_figure.show()

In [ ]:
# ============================================================
# STEP 5F.28 — COMPARE BASELINE AND POST-SHOCK CLUSTERS
# ============================================================

baseline_assignments = (
    cluster_assignments.loc[
        cluster_assignments[
            "period"
        ].astype(str).eq(
            "Baseline_2015_2019"
        ),
        [
            "state_fips",
            "state",
            "Cluster_ID",
            "Cluster_Name"
        ]
    ]
    .rename(
        columns={
            "Cluster_ID": "Baseline_Cluster_ID",
            "Cluster_Name": "Baseline_Cluster_Name"
        }
    )
)

post_shock_assignments = (
    cluster_assignments.loc[
        cluster_assignments[
            "period"
        ].astype(str).eq(
            "Post_Shock_2023_2024"
        ),
        [
            "state_fips",
            "state",
            "Cluster_ID",
            "Cluster_Name"
        ]
    ]
    .rename(
        columns={
            "Cluster_ID": "Post_Shock_Cluster_ID",
            "Cluster_Name": "Post_Shock_Cluster_Name"
        }
    )
)

baseline_post_comparison = (
    baseline_assignments
    .merge(
        post_shock_assignments,
        on=[
            "state_fips",
            "state"
        ],
        how="inner",
        validate="one_to_one"
    )
)

baseline_post_comparison[
    "Baseline_to_Post_Status"
] = np.where(
    baseline_post_comparison[
        "Baseline_Cluster_ID"
    ].eq(
        baseline_post_comparison[
            "Post_Shock_Cluster_ID"
        ]
    ),
    "Same cluster",
    "Changed cluster"
)

baseline_post_comparison[
    "Cluster_Transition"
] = (
    baseline_post_comparison[
        "Baseline_Cluster_ID"
    ].astype(str)
    + " → "
    + baseline_post_comparison[
        "Post_Shock_Cluster_ID"
    ].astype(str)
)

baseline_post_comparison["State_Code"] = (
    baseline_post_comparison["state"]
    .map(STATE_ABBREVIATIONS)
)

assert baseline_post_comparison.shape[0] == 50

display(
    baseline_post_comparison.sort_values(
        [
            "Baseline_to_Post_Status",
            "state"
        ]
    )
)

In [ ]:
# ============================================================
# STEP 5F.29 — SUMMARIZE BASELINE-TO-POST CHANGES
# ============================================================

baseline_post_change_summary = (
    baseline_post_comparison[
        "Baseline_to_Post_Status"
    ]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Number_of_States")
)

baseline_post_change_summary[
    "Percent_of_States"
] = (
    baseline_post_change_summary[
        "Number_of_States"
    ]
    / 50
    * 100
)

display(
    baseline_post_change_summary.round(2)
)

In [ ]:
# ============================================================
# STEP 5F.30 — BASELINE-TO-POST-SHOCK CHANGE MAP
# ============================================================

CHANGE_STATUS_COLORS = {
    "Same cluster": "#A9C4E4",
    "Changed cluster": "#F6BC7B"
}

baseline_post_change_figure = px.choropleth(
    baseline_post_comparison,
    locations="State_Code",
    locationmode="USA-states",
    scope="usa",
    color="Baseline_to_Post_Status",
    color_discrete_map=CHANGE_STATUS_COLORS,
    category_orders={
        "Baseline_to_Post_Status": [
            "Same cluster",
            "Changed cluster"
        ]
    },
    hover_name="state",
    hover_data={
        "State_Code": False,
        "Baseline_Cluster_ID": True,
        "Baseline_Cluster_Name": True,
        "Post_Shock_Cluster_ID": True,
        "Post_Shock_Cluster_Name": True,
        "Cluster_Transition": True,
        "Baseline_to_Post_Status": False
    },
    labels={
        "Baseline_Cluster_ID": "Baseline cluster",
        "Baseline_Cluster_Name": "Baseline profile",
        "Post_Shock_Cluster_ID": "Post-shock cluster",
        "Post_Shock_Cluster_Name": "Post-shock profile",
        "Cluster_Transition": "Transition",
        "Baseline_to_Post_Status": "Comparison"
    }
)

# Mark changed states with an X so color is not the only indicator
changed_states = (
    baseline_post_comparison.loc[
        baseline_post_comparison[
            "Baseline_to_Post_Status"
        ].eq("Changed cluster")
    ]
)

baseline_post_change_figure.add_trace(
    go.Scattergeo(
        locations=changed_states[
            "State_Code"
        ],
        locationmode="USA-states",
        mode="markers",
        marker={
            "symbol": "x",
            "size": 8,
            "color": "#202020",
            "line": {
                "width": 1
            }
        },
        name="Changed-cluster marker",
        hoverinfo="skip"
    )
)

baseline_post_change_figure.update_geos(
    showland=True,
    landcolor="#F4F4F4",
    showlakes=True,
    lakecolor="white",
    showsubunits=True,
    subunitcolor="white",
    subunitwidth=1.2
)

baseline_post_change_figure.update_layout(
    title={
        "text": (
            "State Cluster Membership: Baseline vs. Post-Shock"
            "<br>"
            "<sup>"
            "An X identifies a state assigned to a different cluster "
            "in 2023–2024"
            "</sup>"
        ),
        "x": 0.5,
        "xanchor": "center"
    },
    width=1300,
    height=700,
    margin={
        "l": 20,
        "r": 260,
        "t": 110,
        "b": 30
    },
    legend={
        "title": {
            "text": "Baseline-to-post comparison"
        },
        "x": 1.01,
        "y": 1
    },
    paper_bgcolor="white"
)

baseline_post_change_figure.show()

In [ ]:
# ============================================================
# STEP 5F.31 — SAVE MAPS AND COMPARISON DATA
# ============================================================

post_shock_cluster_figure.write_html(
    FIGURES_DIR
    / "post_shock_economic_structure_clusters.html",
    include_plotlyjs=True
)

period_comparison_figure.write_html(
    FIGURES_DIR
    / "three_period_cluster_map_comparison.html",
    include_plotlyjs=True
)

baseline_post_change_figure.write_html(
    FIGURES_DIR
    / "baseline_to_post_shock_cluster_change_map.html",
    include_plotlyjs=True
)

baseline_post_comparison.to_csv(
    TABLES_DIR
    / "baseline_to_post_shock_cluster_comparison.csv",
    index=False
)

baseline_post_change_summary.to_csv(
    TABLES_DIR
    / "baseline_to_post_shock_change_summary.csv",
    index=False
)

print("Post-shock and comparison maps saved successfully.")